<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/Global_bathy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap


In [ ]:
# 1. Authenticate and Initialize Earth Engine
# Run ee.Authenticate() if it's your first time running GEE in this session
try:
    ee.Initialize(project='ee-jaalvalcan')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-jaalvalcan')

In [ ]:
# 2. Define the path to your Lake Tanganyika shapefile
shp_path = '/content/sample_data/Tanga.shp'

# Install pycrs if not already installed, as it's often a dependency for geemap.shp_to_ee
!pip install pycrs

# 3. Convert the local shapefile to an Earth Engine object
roi = geemap.shp_to_ee(shp_path)

# 4. Load the GLOBathy global lakes bathymetry dataset
globathy = ee.Image("projects/sat-io/open-datasets/GLOBathy/GLOBathy_bathymetry")

# 5. Clip the bathymetry dataset to Lake Tanganyika's borders
clipped_globathy = globathy.clip(roi)

# 6. Define 10 depth classes optimized for Tanganyika (from 1m up to 1500m)
# This creates a sequence: 1, 151, 301, 451, 601, 751, 901, 1051, 1201, 1351
thresholds = ee.List.sequence(1, 1500, 150)

# Create the classified image by iterating through the depth thresholds
classified = clipped_globathy.gt(ee.Image.constant(thresholds.get(0)))
for i in range(1, 10):
    classified = classified.add(clipped_globathy.gt(ee.Image.constant(thresholds.get(i))))

# 7. Define a striking visualization palette (Deep Blue for maximum depth)
vis_params = {
    'min': 0,
    'max': 9,
    'palette': [
        '#ffff00', '#ccff66', '#99ff99', '#66ffcc', '#00ffff',
        '#00ccff', '#0099ff', '#0066ff', '#0033cc', '#0000ff'
    ]
}

# 8. Create and display the interactive map
Map = geemap.Map()
Map.centerObject(roi, 7) # Dynamically zoom and center on Lake Tanganyika
Map.addLayer(classified, vis_params, "Lake Tanganyika Bathymetry (10 Classes)")

# Add a legend to the map using actual depth values for the scale
legend_labels = []
for j in range(10):
    current_threshold_value = thresholds.get(j).getInfo()
    next_threshold_value = thresholds.get(j+1).getInfo() if j < 9 else 1500

    if j == 0:
        legend_labels.append(f'< {next_threshold_value} m')
    elif j == 9:
        legend_labels.append(f'> {current_threshold_value} m')
    else:
        legend_labels.append(f'{current_threshold_value} - {next_threshold_value} m')

# Update color bar to showcase data range (0-1500) instead of class index (0-9)
Map.add_colorbar(
    vis_params['palette'],
    legend_labels,
    orientation='vertical',
    label='Depth (m)',
    vmin=0,
    vmax=1500
)

# Render the map in the Colab output
Map